# GPU latency and FPS for the MoR detection paper

Runs `paper/analysis/latency.py` on a GPU for all seven models in the paper: RT-DETR 2L / 6L, RT-DETR-MoR 2x3 / 3x2, DETR, and DETR-MoR cyclic / sequential.

- **Inputs:** batch-1 latency and batched throughput at 640x640, in fp32 and fp16 autocast.
- **Weights:** random. Latency does not depend on them, and the MoR routers keep a fixed number of tokens per recursion. **No checkpoints and no dataset are needed**, only the repository code and configs.

**Kaggle.** Either
- add the repository as a Dataset, uploading the repo folder without the `.pth` files but with the `rt_detr_all_results/**/**.yaml` configs; or
- set `GIT_URL` below.

Then enable a GPU accelerator and *Run all*.

**Laptop.** Open this notebook from `paper/analysis/` inside the repo and *Run all*.

**Output.** `latency_gpu_<gpu>.json` plus `profile_rtdetr_<gpu>_*.json` in the working directory. Copy them to `paper/analysis/out/` in the repo.

In [ ]:
import glob, json, os, platform, re, subprocess, sys

GIT_URL = ''            # e.g. 'https://github.com/<user>/detr-mor.git' (optional)
GIT_BRANCH = 'paper-nca'
ITERS, WARMUP = 200, 50
BATCH_SIZES = [1, 8]
PRECISIONS = ['fp32', 'fp16']

def find_repo():
    here = os.path.abspath(os.getcwd())
    for cand in (here, os.path.join(here, '..'), os.path.join(here, '..', '..')):
        if os.path.isfile(os.path.join(cand, 'paper', 'analysis', 'latency.py')):
            return os.path.abspath(cand)
    for cand in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
        if os.path.isfile(os.path.join(cand, 'paper', 'analysis', 'latency.py')):
            return cand
    if GIT_URL:
        dest = os.path.abspath('detr-mor')
        if not os.path.isdir(dest):
            subprocess.run(['git', 'clone', '--depth', '1', '-b', GIT_BRANCH, GIT_URL, dest], check=True)
        return dest
    raise FileNotFoundError('repository not found: add it as a Kaggle dataset, set GIT_URL, '
                            'or run this notebook from paper/analysis inside the repo')

REPO = find_repo()
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(REPO, 'paper', 'analysis', 'out')
os.makedirs(OUT_DIR, exist_ok=True)
print('repo:', REPO, '| outputs:', OUT_DIR)

In [ ]:
# Dependencies (Kaggle images already have torch, torchvision, scipy, yaml)
for mod, pkg in (('yaml', 'pyyaml'), ('scipy', 'scipy'), ('torchvision', 'torchvision'), ('pandas', 'pandas')):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU visible: enable an accelerator (Kaggle: Settings > Accelerator)'
GPU = torch.cuda.get_device_name(0)
TAG = re.sub(r'[^A-Za-z0-9]+', '_', GPU).strip('_').lower()
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU:', GPU)
subprocess.run(['nvidia-smi'])

## End-to-end latency and FPS (all seven models)

Timed with CUDA events around the full forward pass, including post-processing to detections. Encoder and decoder stage times come from forward hooks.

- **FPS** is `1000 * batch / median_ms`.
- **Batch 1** is the real-time latency setting; **batch 8** gives throughput.

In [ ]:
latency_json = os.path.join(OUT_DIR, f'latency_gpu_{TAG}.json')
cmd = [sys.executable, os.path.join(REPO, 'paper', 'analysis', 'latency.py'),
       '--device', 'cuda', '--iters', str(ITERS), '--warmup', str(WARMUP),
       '--batch-sizes', *map(str, BATCH_SIZES), '--precisions', *PRECISIONS,
       '--out', latency_json]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
data = json.load(open(latency_json))
rows = [{'model': r['name'], 'precision': r['precision'], 'batch': r['batch'],
         'median ms': round(r['ms_median'], 2), 'p90 ms': round(r['ms_p90'], 2),
         'FPS': round(r['fps'], 1),
         **{f'{k} ms': round(v, 2) for k, v in r['stage_ms_median'].items()}}
        for r in data['results']]
df = pd.DataFrame(rows)
print(data['hardware'], '| torch', data['torch'])
df

## Per-stage breakdown of the RT-DETR family (batch 1)

`scripts/profile_latency.py` splits the forward pass into backbone, input projection, AIFI, CCFM, query selection, decoder and heads. This shows how much of the latency the (recursive) decoder accounts for.

In [ ]:
R = 'rt_detr_all_results'
RT = {'2L': ('rt_detr.ipynb', f'{R}/rt_detr_voc_results (2 blocks)/rt_detr_config.yaml'),
      '6L': ('rt_detr.ipynb', f'{R}/rt_detr_new_results (6 blocks)/rt_detr_config.yaml'),
      '2x3': ('rt_detr_mor.ipynb', 'rt_detr_mor_config.yaml'),
      '3x2': ('rt_detr_mor.ipynb', f'{R}/rt_detr_mor_voc_results (3 blocks, 2 recursions)/rt_detr_mor_config.yaml')}
for name, (nb, cfg) in RT.items():
    for prec in PRECISIONS:
        out = os.path.join(OUT_DIR, f'profile_rtdetr_{TAG}_{name}_{prec}.json')
        cmd = [sys.executable, os.path.join(REPO, 'scripts', 'profile_latency.py'),
               '--notebook', os.path.join(REPO, nb), '--config', os.path.join(REPO, cfg),
               '--sizes', '640', '--num-queries', '25', '--iters', str(ITERS),
               '--warmup', str(WARMUP), '--out', out] + (['--fp16'] if prec == 'fp16' else [])
        print(f'== {name} {prec}')
        subprocess.run(cmd, check=True, cwd=REPO)

## Done

Copy these files from the output directory into `paper/analysis/out/` of the repository:
- `latency_gpu_<gpu>.json`
- `profile_rtdetr_<gpu>_*.json`

On Kaggle, use the *Output* panel to download them.

In [ ]:
for f in sorted(glob.glob(os.path.join(OUT_DIR, f'*{TAG}*.json'))):
    print(f)